it is also called as the thread memory

The history for the frontend is managed in seperate backed. In thread memory we implement this for proper AI response maintaing context.

We properly store the context by using n previous messages and history of all other messages older than last n messages

In [143]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_core.messages import trim_messages, BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import RemoveMessage
from langchain_core.messages.utils import count_tokens_approximately

In [144]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",  # Your Azure deployment name
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1",
    )

MAX_TOKENS = 150

In [145]:
class State(TypedDict):
    Human: str
    AI: str
    summary: str
    messages: Annotated[list[BaseMessage], operator.add]

In [146]:
def chat_node(state: State):
    messages = list(state.get("messages", []))
    messages.append(HumanMessage(content=state["Human"]))
    
    trimmed_messags= trim_messages(
    messages=messages,
    max_tokens=MAX_TOKENS,
    strategy="last",
    token_counter=count_tokens_approximately,
)
    print("Tokens count", count_tokens_approximately(messages=trimmed_messags))
    response = llm.invoke(trimmed_messags)
    ai_response = AIMessage(content = response.content)
    to_be_return = [HumanMessage(state['Human']), ai_response]
    return {
        "messages": to_be_return,
        "AI":ai_response
    }

    

In [147]:
def summary_node(state : State):
    messages = list(state.get("messages", []))
    summary = state['summary']
    if summary:
        prompt = (
            f"Extend the summary {state['summary']} using the new messages: {state['messages']}"
        )
    else:
        prompt=(
            f"Generate summary of these messages {state['messages']}"
        )
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    print(response.content)

    messages_to_delete = [
        RemoveMessage(id=msg.id)
        for msg in messages[:4]
        ]
    
    return {
        "summary": response.content,
        "messages":messages_to_delete
    }

def need_summary(state:State):
    return len(state['messages'])>6



    

In [148]:
graph = StateGraph(State)
graph.add_node('chat', chat_node)
graph.add_node('summary', summary_node)
graph.add_edge(START, 'chat')
graph.add_conditional_edges('chat', need_summary, 
                            {
                                True:'summary',
                                False:END
                            }
)
graph.add_edge('summary', END)
memory_saver = InMemorySaver()
workflow = graph.compile(checkpointer=memory_saver)

config = {
    "configurable":{
        "thread_id":"nadeem"
    }
}



In [149]:
for i in range(1,10):
    initial_state = {
    "Human":f"what is {i}+ {i}-1",
    "summary":""}

    execution = workflow.invoke(initial_state,
                            config=config)
    print(execution['messages'])
    print(execution['summary'])


Tokens count 8
[HumanMessage(content='what is 1+ 1-1', additional_kwargs={}, response_metadata={}), AIMessage(content='1 + 1 - 1 = 1', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

Tokens count 25
[HumanMessage(content='what is 1+ 1-1', additional_kwargs={}, response_metadata={}), AIMessage(content='1 + 1 - 1 = 1', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is 2+ 2-1', additional_kwargs={}, response_metadata={}), AIMessage(content='2 + 2 - 1 = 3', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

Tokens count 42
[HumanMessage(content='what is 1+ 1-1', additional_kwargs={}, response_metadata={}), AIMessage(content='1 + 1 - 1 = 1', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is 2+ 2-1', additional_kwargs={}, response_metadata={}), AIMessage(content='2 + 2 - 1 = 3', additional_kwargs={},

ValueError: Unknown BaseMessage type <class 'langchain_core.messages.modifier.RemoveMessage'>.